DSA Week04 - 前中后缀表达式 (Prefix/Infix/Postfix Expressions)
来源: 202603_DSA_W04_Complexity_LinearStructures
调度场算法 Shunting Yard 是重点

三种表达式对比:
+-----------------+-----------------+-----------------+
| Infix (中缀)    | Prefix (前缀)   | Postfix (后缀)  |
+-----------------+-----------------+-----------------+
| A + B           | + A B           | A B +           |
| A + B * C       | + A * B C       | A B C * +       |
| (A + B) * C     | * + A B C       | A B + C *       |
| A + B * C + D   | + + A * B C D   | A B C * + D +   |
| (A + B) * (C+D) | * + A B + C D   | A B + C D + *   |
| A * B + C * D   | + * A B * C D   | A B * C D * +   |
| A + B + C + D   | + + + A B C D   | A B + C + D +   |
+-----------------+-----------------+-----------------+

关键原则:
- 前缀/后缀不需要括号，操作顺序完全由操作符位置决定
- 操作数的相对顺序在三种表示法中保持不变
- 只有中缀需要额外的括号和优先级规则

1. 中缀转后缀 - 调度场算法 (Shunting Yard)
   适用于单字符操作数 (A, B, C... 或 0-9)
优先级管理: * , / > + , - > (
处理规则:
  1. 操作数直接输出
  2. 遇到 ( 入栈
  3. 遇到 ) 弹出栈顶直至遇到 (
  4. 遇到运算符，弹出栈中优先级更高或相等的运算符，再将当前操作符入栈

In [ ]:
def infixToPostfix(infixexpr):
    """
    中缀转后缀 (操作数以空格分隔的字符串)
    例: "A * B + C * D" -> "A B * C D * +"
    """
    prec = {"*": 3, "/": 3, "+": 2, "-": 2, "(": 1}
    opStack = []
    postfixList = []
    tokenList = infixexpr.split()

    for token in tokenList:
        if token in "ABCDEFGHIJKLMNOPQRSTUVWXYZ" or token in "0123456789":
            postfixList.append(token)
        elif token == '(':
            opStack.append(token)
        elif token == ')':
            topToken = opStack.pop()
            while topToken != '(':
                postfixList.append(topToken)
                topToken = opStack.pop()
        else:
            while opStack and (prec[opStack[-1]] >= prec[token]):
                postfixList.append(opStack.pop())
            opStack.append(token)

    while opStack:
        postfixList.append(opStack.pop())

    return " ".join(postfixList)

2. 中缀转后缀 - 支持浮点数 (number buffer 技巧)
   OJ24591: 中序表达式转后序表达式
   http://cs101.openjudge.cn/practice/24591/

In [ ]:
def infix_to_postfix(expression):
    """
    中缀转后缀 (支持多位数和浮点数，无空格分隔)
    例: "3+4.5*(7+2)" -> "3 4.5 7 2 + * +"
    """
    precedence = {'+': 1, '-': 1, '*': 2, '/': 2}
    stack = []
    postfix = []
    number = ''

    for char in expression:
        if char.isnumeric() or char == '.':
            number += char  # 累积数字字符
        else:
            if number:
                num = float(number)
                postfix.append(int(num) if num.is_integer() else num)
                number = ''
            if char in '+-*/':
                while stack and stack[-1] in '+-*/' and \
                        precedence[char] <= precedence[stack[-1]]:
                    postfix.append(stack.pop())
                stack.append(char)
            elif char == '(':
                stack.append(char)
            elif char == ')':
                while stack and stack[-1] != '(':
                    postfix.append(stack.pop())
                stack.pop()  # 弹出 '('

    if number:
        num = float(number)
        postfix.append(int(num) if num.is_integer() else num)

    while stack:
        postfix.append(stack.pop())

    return ' '.join(str(x) for x in postfix)

2b. 中缀转后缀 - 用正则表达式分词 (re版)

In [ ]:
def infix_to_postfix_re(expression):
    """使用正则表达式拆分 tokens"""
    import re
    tokens = re.split(r'([\(\)\+\-\*\/])', expression)
    tokens = [item for item in tokens if item.strip()]
    return tokens  # 返回 token 列表供后续处理

3. 后缀表达式求值 (Postfix Evaluation)
   核心: 遇到操作数入栈; 遇到运算符弹出两个数运算，结果重新入栈

版本1: 单字符操作数

In [ ]:
def postfixEval(postfixExpr):
    """
    后缀表达式求值 (操作数为单个数字)
    例: '7 8 + 3 2 + /' -> 3.0
    """
    operandStack = []
    tokenList = postfixExpr.split()

    for token in tokenList:
        if token in "0123456789":
            operandStack.append(int(token))
        else:
            operand2 = operandStack.pop()
            operand1 = operandStack.pop()
            result = doMath(token, operand1, operand2)
            operandStack.append(result)

    return operandStack.pop()


def doMath(op, op1, op2):
    if op == "*":
        return op1 * op2
    elif op == "/":
        return op1 / op2
    elif op == "+":
        return op1 + op2
    else:
        return op1 - op2

版本2: 支持浮点数
OJ24588: 后序表达式求值
http://cs101.openjudge.cn/practice/24588/

In [ ]:
def evaluate_postfix(expression):
    """
    后缀表达式求值 (支持浮点数)
    例: '5 3.4 + 6 /' -> 1.4
    步骤:
      1. 从左到右扫描
      2. 遇到数字压入栈中
      3. 遇到运算符，弹出两个数: 先弹出的是右操作数，后弹出的是左操作数
      4. 运算结果压回栈中
      5. 扫描完毕，栈顶即为结果
    """
    stack = []
    tokens = expression.split()

    for token in tokens:
        if token in '+-*/':
            right_operand = stack.pop()
            left_operand = stack.pop()
            if token == '+':
                stack.append(left_operand + right_operand)
            elif token == '-':
                stack.append(left_operand - right_operand)
            elif token == '*':
                stack.append(left_operand * right_operand)
            elif token == '/':
                stack.append(left_operand / right_operand)
        else:
            stack.append(float(token))

    return stack[0]

附: 栈使用的重要原则 (来自课件)
1. 栈的基本用途是"等待"。当一条指令的操作依赖于后续指令时，
   先将其入栈等待，了解到具体操作后再 pop 处理。

2. 栈与括号匹配关系密切。遇到左括号时入栈，遇到右括号时
   回头找到匹配的左括号并处理。

3. 观察示例找规律: 在中序转后序中，数字顺序不变，运算符
   总是在运算单元最后。因此数字不入栈直接输出，运算符入栈
   等待运算单元结束后弹出。

测试代码

In [ ]:
if __name__ == "__main__":
    print("=== 中缀转后缀 (单字符操作数) ===")
    print(infixToPostfix("A * B + C * D"))
    # A B * C D * +
    print(infixToPostfix("( A + B ) * C - ( D - E ) * ( F + G )"))
    # A B + C * D E - F G + * -
    print(infixToPostfix("( A + B ) * ( C + D )"))
    # A B + C D + *
    print(infixToPostfix("( A + B ) * C"))
    # A B + C *
    print(infixToPostfix("A + B * C"))
    # A B C * +

    print("\n=== 中缀转后缀 (浮点数) ===")
    print(infix_to_postfix("7+8.3"))
    # 7 8.3 +
    print(infix_to_postfix("3+4.5*(7+2)"))
    # 3 4.5 7 2 + * +
    print(infix_to_postfix("(3)*((3+4)*(2+3.5)/(4+5))"))
    # 3 3 4 + 2 3.5 + * 4 5 + / *

    print("\n=== 后缀表达式求值 (单字符) ===")
    print(postfixEval('7 8 + 3 2 + /'))
    # 3.0

    print("\n=== 后缀表达式求值 (浮点数) ===")
    print(f"{evaluate_postfix('5 3.4 +'):.2f}")
    # 8.40
    print(f"{evaluate_postfix('5 3.4 + 6 /'):.2f}")
    # 1.40
    print(f"{evaluate_postfix('5 3.4 + 6 * 3 +'):.2f}")
    # 53.40

OJ 题目模板: 批量读取输入
--- 中序转后序 (OJ24591) ---
n = int(input())
for _ in range(n):
    expression = input()
    print(infix_to_postfix(expression))

--- 后序表达式求值 (OJ24588) ---
n = int(input())
for _ in range(n):
    expression = input()
    result = evaluate_postfix(expression)
    print(f"{result:.2f}")